# Model inference - generate submission.csv

Loads the best DeepCNN checkpoint (deep_cnn_best.pt, saved by the DeepCNN notebook), predicts on the competition test.csv, and writes the submission. Run the DeepCNN notebook first so the checkpoint exists in the session.

## 1. Setup

In [ ]:
# --- install + imports ---
!pip -q install wandb
import os, math, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
torch.manual_seed(42); np.random.seed(42)

## 2. Load test set

In [ ]:
# --- load the competition test set (test.csv has pixels only, no labels) ---
IMG_SIZE, NUM_CLASSES = 48, 7
FER_MEAN, FER_STD = 0.507, 0.255
def parse_pixels(s):
    return np.fromstring(s, dtype=np.float32, sep=" ").reshape(IMG_SIZE, IMG_SIZE)
tf = transforms.Compose([transforms.ToPILImage(), transforms.ToTensor(),
                         transforms.Normalize([FER_MEAN], [FER_STD])])
df = pd.read_csv("data/test.csv"); df.columns = [c.strip() for c in df.columns]
imgs = np.stack([parse_pixels(p) for p in df["pixels"].values])
print("test images:", imgs.shape)

## 3. Re-declare the DeepCNN architecture

In [ ]:
# === Iteration 3: Deep CNN + BatchNorm + Dropout (-> expected GOOD FIT) ===
def vgg_stage(i, o, dropout=0.0):
    layers = [nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
              nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
              nn.MaxPool2d(2)]
    if dropout > 0: layers.append(nn.Dropout2d(dropout*0.5))
    return nn.Sequential(*layers)

class DeepCNN(nn.Module):
    """3 VGG stages + global average pool + regularized head. Depth gives capacity;
    BN + dropout + (weight decay/augmentation) give generalization."""
    def __init__(self, channels=(64,128,256), fc=256, dropout=0.4):
        super().__init__()
        c1, c2, c3 = channels
        self.features = nn.Sequential(vgg_stage(1,c1,dropout),
                                      vgg_stage(c1,c2,dropout),
                                      vgg_stage(c2,c3,dropout))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(c3, fc),
                                        nn.BatchNorm1d(fc), nn.ReLU(),
                                        nn.Dropout(dropout), nn.Linear(fc, NUM_CLASSES))
    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

## 4. Predict + write submission

In [ ]:
# --- load best checkpoint, predict, write submission.csv ---
model = DeepCNN().to(device)
model.load_state_dict(torch.load("deep_cnn_best.pt", map_location=device))
model.eval()

preds = []
with torch.no_grad():
    for i in range(0, len(imgs), 256):
        batch = torch.stack([tf(imgs[j].astype(np.uint8))
                             for j in range(i, min(i+256, len(imgs)))]).to(device)
        preds += model(batch).argmax(1).cpu().tolist()

with open("submission.csv", "w") as f:
    for p in preds: f.write(f"{p}\n")
print("wrote", len(preds), "predictions to submission.csv")